In my opinion, pipelines in scikit-learn are one of the most useful things in the library. They allow you to sequentially apply a list of transformers and a final estimator to your data. Furthermore, they can be used in cross-validation. This ensures that data transformations within the cross-validation loop are only fitted on the training data. This prevents data leakage and better generalization on the test dataset or in production.

Pipelines can also be used in a  grid or random search for the best hyper-parameters. However, this is not limited to model parameters. We can also search for the best data transformations hyper-parameters (imputation with the median or the mean) in conjunction with the model hyper-parameters.

::: {.column-margin}
::: {.callout-note}
We use a house pricing [dataset](https://www.kaggle.com/c/house-prices-advanced-regression-techniques) from kaggle for demonstrating the pipelines.
:::
:::

However, in this post, I look at the fundamentals of the pipeline. We focus on implementing scikit-learn transformers and custom transformers into a scikit-learn pipeline. How do we apply different transformations to different data types or features? After implementing this, we can save the entire data pipeline (data transformations and the estimator) into a single artifact. This is great when we want to use the model in production because we need to deal with one file.


In [16]:
import pandas as pd

from sklearn.model_selection import train_test_split

# Load data
data = pd.read_csv("data/house-prices.csv")

# Features and target select
X = data.drop(columns=["SalePrice"])
y = data["SalePrice"]

# Splitting
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

# Display first two rows of features
X_train.head(1).iloc[:, :5]

,Id,MSSubClass,MSZoning,LotFrontage,LotArea
921,922,90,RL,67.0,8777


## Pipelines

Scikit-learn's pipeline object expects a list of steps. These steps need to be tuples containing a name and a scikit-learn transformer (name, transform). Let's first create a simple pipeline that fills missing numerical values with the mean. This can be done with the `SimpleImputer()` class from scikit-learn.


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Numerical single transformer pipeline
num_pipeline = Pipeline(steps=[
  ("num_imputer",  SimpleImputer(strategy="mean")),
])

# Fit numerical transformer pipeline
num_pipeline.fit(X_train[["LotFrontage"]])

# Display first five transformed rows
num_pipeline.transform(X_test[["LotFrontage"]])[:5]

array([[80.        ],
       [60.        ],
       [70.21063608],
       [21.        ],
       [70.21063608]])

In the preceding code block, we created a pipeline that imputes numerical values and fills them with the mean. The pipeline object can be used to fit and transform data just as you are used to when using a single transformer.

Let us now chain two transformers together into a single pipeline object. For example, after imputing the missing data and filling it with the mean, we want to scale the data between 0 and 1. We can use the `MinMaxScaler()` to do this scaling.

In [3]:
from sklearn.preprocessing import MinMaxScaler

# Numerical multi transformer pipeline
num_pipeline = Pipeline(steps=[
  ("num_imputer",  SimpleImputer(strategy="mean")),
  ("scaler", MinMaxScaler(feature_range=(0, 1)))
])

# Fit numerical transformer pipeline
num_pipeline.fit(X_train[["LotFrontage"]])

# Display first five transformed rows
num_pipeline.transform(X_test[["LotFrontage"]])[:5]

array([[0.20205479],
       [0.13356164],
       [0.16852958],
       [0.        ],
       [0.16852958]])

We have now chained to transformers together and can be fitted on the training data and transform testing data. We can also serialize this entire pipeline if we want to use it later in production.

## Data types and Pipeline Transformers

Until this point, we only transformed numerical data. How do we deal with a table containing columns with numerical values and columns containing categorical data? Do we need several separate pipelines? This can become complicated quickly. However, scikit-learn provides a nice and simple solution to this problem: the column transformer. This transformer can apply different preprocessing and feature extraction pipelines to different subsets of features. Meaning that we can use other preprocessing steps to transform the numerical data than the transformations used to process the categorical data.

So how do we do this? First, we create a pipeline for numerical data transformers. These transformers will be applied linearly (the order that the transformation steps are implemented). Secondly, we create a transformation pipeline for categorical data. We can now use the ColumnTransformer(), which takes a list of transformers as tuples (name, transformer pipeline, and the columns to transform) and apply the transformations.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Pipeline of numerical transformers
num_transformer_pipeline = Pipeline(steps=[
  ("num_imputer",  SimpleImputer(strategy="mean")),
  ("scaler", MinMaxScaler(feature_range=(0, 1)))
])

# Pipeline of categorical transformers
cat_transformer_pipeline = Pipeline(steps=[
  ("cat_imputer", SimpleImputer(strategy="most_frequent")),
  ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Create preprocessing pipeline
preprocessors = ColumnTransformer(transformers=[
  ("num_transformer", num_transformer_pipeline, ["LotFrontage"]),
  ("cat_transformer", cat_transformer_pipeline, ["Street", "Alley"]),
])

# Fit transformer pipeline
preprocessors.fit(X_train)

# Display first three transformed rows
preprocessors.transform(X_test)[:3, :]

array([[0.20205479, 0.        , 1.        , 1.        , 0.        ],
       [0.13356164, 0.        , 1.        , 1.        , 0.        ],
       [0.16852958, 0.        , 1.        , 1.        , 0.        ]])

As you can see in the diagram, we have two transformer pipelines that are combined into a single pipeline object. The numerical transformer pipeline is applied to the `LotFrontage` column of the table. In parallel, the categorical transformer pipeline is applied to the `Street` and `Alley` columns. Basically, we can create data processing pipelines for every individual feature in our dataset.

However, we do not necessarily need to specify the columns to transform. We can select them automatically using `make_column_selector()`. This class can select columns based on the data type or the column name.


In [5]:
from sklearn.compose import make_column_selector

# Create preprocessing pipeline
preprocessors = ColumnTransformer(transformers=[
  ("num_transformer", num_transformer_pipeline, make_column_selector(dtype_include="number")),
  ("cat_transformer", cat_transformer_pipeline, make_column_selector(dtype_include="category")),
])

# Fit transformer pipeline
preprocessors.fit(X_train)

# Display first transformed row
X_test = preprocessors.transform(X_test)[0, :]

In the preceding code block, we use the make_column_selector to only include select all the numerical ("number") columns and then to select all the categorical columns ("category"). This automates selecting the columns and applying the different transformations to the data. This single pipeline can be serialized. This makes it easy to use data transformers in production because we only need to load the preprocessing pipeline to call `transform()` to process the new data.

To complete the pipeline, you can add an estimator to the end of the pipeline. This is done by creating another pipeline instance where you combine the data processing pipeline with an estimator.


In [10]:
from sklearn.linear_model import LinearRegression

# Preprocessing transformers and estimator
regressor_pipeline = Pipeline(steps=[
  ("preprocessors", preprocessors),
  ("regressor", LinearRegression())
])

# Fit full pipeline
regressor_pipeline.fit(X_train, y_train);

This pipeline (transformers + estimators) can be used in all kinds of cross-validation or random/grid searches.

## Custom Transformers

Even though scikit-learn provides a lot of transformers, you may find that you want to transform your data in a way that is not implemented in scikit-learn. In this case, you need to create a custom transformer. This can be done by inheriting from the `BaseEstimator()` and `TransformerMixin()` classes. You only need to implement your own `fit()` and `transform()` or `predict()` methods.

Let's say we want to do frequency encoding. Meaning that every category in a categorical column is encoded using the frequency of occurring. Meaning "Category A" occupies 20% of all the values in a categorical variable, the "Category A" should equal 0.2. When we encounter a category, we have not seen during training, we set the frequency of occurrence of this particular category to 0.

In [7]:
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin

class FrequencyEncoding(BaseEstimator, TransformerMixin):
    def __init__(self, normalize=True):
        # Initialize frequency dictionary
        self.freq_dict = {}
        self.normalize = normalize

    def fit(self, X, y=None):
        # Check if X is pandas Series or DataFrame
        if isinstance(X, pd.Series) or isinstance(X, pd.DataFrame):

            # Convert to numpy array
            X = X.values

        # 1. Loop over columns
        for i, feature in enumerate(X.T):

            # 2. Calculate (normalized) frequency
            counts = pd.Series(feature).value_counts(normalize=self.normalize).to_dict()

            # 3. Store counts in dictionary
            self.freq_dict[i] = counts

        return self
  
    def transform(self, X):
        # Check if X is pandas Series or DataFrame
        if isinstance(X, pd.Series) or isinstance(X, pd.DataFrame):

            # Convert to numpy array
            X = X.values

        # 4. Loop over columns
        for i, feature in enumerate(X.T):

            # 5. Convert to series
            series = pd.Series(X.T[i])

            # 6. Check if values in frequency dict
            condition = series.isin(self.freq_dict[i])

            # 7. If present fill with value, otherwise with zero
            encoded_values = np.where(condition, series.map(self.freq_dict[i]), 0)

            # 8. Store back into original matrix
            X.T[i] = encoded_values

        return X

The code above implements a frequency encoder. Because we used the base classes from scikit-learn this transformer can now be used in a pipeline.

### Training

1. Loop over the columns of the table/matrix.
2. Count the values/categories in the columns and if we want to normalize them.
3. Store values/categories and corresponding (normalized) frequency into a dictionary with the categories as keys and the count as values.

### Transforming

1. Loop over the columns of the table/matrix.
2. Convert array to pandas series.
3. Check if the categories in the column are in the dictionary (keys).
4. If they are, map the values in the directory to the corresponding category (keys) to the values in the column. If the category value is not in the dictionary, fill with a 0.
5. Store the encoded values back into the original matrix.

Let's use the same pipeline we did earlier. However, now we switch out the `OneHotEncoder()` out for our own custom `FrequencyEncoder()`.

In [18]:
# Pipeline of numerical transformers
num_transformer_pipeline = Pipeline(steps=[
  ("num_imputer",  SimpleImputer(strategy="mean")),
  ("scaler", MinMaxScaler(feature_range=(0, 1)))
])

# Pipeline of categorical transformers
cat_transformer_pipeline = Pipeline(steps=[
  ("cat_imputer", SimpleImputer(strategy="most_frequent")),
  ("freq_encoder", FrequencyEncoding(normalize=True)) # Custom transformer
])

# Create preprocessing pipeline
preprocessors = ColumnTransformer(transformers=[
  ("num_transformer", num_transformer_pipeline, make_column_selector(dtype_include="number")),
  ("cat_transformer", cat_transformer_pipeline, make_column_selector(dtype_include="category")),
])

regressor_pipeline = Pipeline(steps=[
  ("preprocessors", preprocessors),
  ("regressor", LinearRegression())
])

We can now fit the entire pipeline containing our custom transformer on the dataset. Furthermore, we can use the `score()` method or perform cross-validation. We can also persist the entire pipeline to make use of it later on in production. We only need to load our persisted model pipeline, and call the `predict()` method on new data.


In [20]:
regressor_pipeline.fit(X_train, y_train)
score = regressor_pipeline.score(X_test, y_test)
score

0.8143374311323394

## Conclusion

In this post, I wanted the demonstrate how powerful and useful the scikit-learn's Pipeline API is. We can easily implement any data transformation or model estimator leveraging some scikit-learn base functionality. We can then persist the whole data transforming and modelling pipeline into a single serialized artifact, which is great when we want to use this model later on.

The pipeline demonstrated in this post is really simple, and only scratches the surface of what is possible when using them.